In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "04-inference-engine/vllm-internals/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Block hashes, eviction and admission: what vLLM does on top of the textbook cache

**Tier:** T0 (Python standard library only; laptop or Colab CPU). The last section also imports the serving lab's
`servelab.sizing` from this repository (standard library too) and skips those checks if the lab is not present.

**The one-minute version.** vLLM names every full KV block by a hash chained over the whole prefix, keeps freed
blocks cached in a doubly linked free list, and allocates from that list's head: uncached blocks first, then cached
blocks in LRU order, each prefix chain's tail before its root. Admission is a capacity check over the *whole*
current sequence in which cached hits that sit in the free list count as blocks to be taken. After this notebook
you should be able to predict the free-list order after any sequence of frees, say how many tokens of a prompt a
cache hit saves, and explain why a preempted request often cannot come back until the request that displaced it
finishes (primer sections 3.6, 3.7 and 4.3–4.6 of [`../vllm-internals-primer.md`](../vllm-internals-primer.md);
vLLM `main` at `5840d95`, 2026-09-25).

**Read first, if the idea is new:** the repo's mini engine builds the same cache from scratch in
[`../../serving-engine/mini-engine-core/minengine/kv.py`](../../serving-engine/mini-engine-core/minengine/kv.py)
(notebook `03_prefix_caching`). This notebook is about what vLLM does **differently**, and what follows from it:

| Here | vLLM (`vllm/v1/core/`, `vllm/utils/hashing.py`) | mini engine (`minengine/kv.py`) |
|---|---|---|
| `H`, `NONE_HASH` | `hashing.sha256` (pickle, then SHA-256); `kv_cache_utils.init_none_hash` (fixed seed) | `hash_block` (SHA-256 of `repr`), parent `None` |
| `block_hashes(tokens, lora_name, cache_salt)` | `hash_block_tokens`, `get_request_block_hasher`, `generate_block_hash_extra_keys`: LoRA name on every block, salt on the first only | `block_hashes(tokens, block_size, extra)`: one `extra` on every block |
| `FreeQueue` | `FreeKVCacheBlockQueue` (intrusive doubly linked list, O(1) `remove`) | `OrderedDict` of block ids |
| `BlockPool.null_block` | block 0 reserved; `get_usage = 1 − free / (num_blocks − 1)` | no reserved block |
| `free_blocks` (exercise 3) | `BlockPool.free_blocks`: uncached blocks to the **head**, cached to the tail | every block to the back |
| `longest_hit` (exercise 1) | `KVCacheManager.get_computed_blocks` → `find_longest_cache_hit`, capped at `num_tokens − 1` | `lookup`, same cap |
| `can_admit` (exercise 4) | `allocate_slots(..., full_sequence_must_fit=True)` → `get_num_blocks_to_allocate` | `allocate_slots(..., admit_whole_prompt=True)` |
| `admit` caches the full blocks at allocation | `allocate_slots` → `cache_blocks` at **schedule time** | `cache_blocks` after the step |

Simplifications: one KV cache group, no sliding window, no multimodal keys, and a block size of 4 instead of 16 so the
printouts stay short.

**How the exercises work.** Each exercise cell holds a stub that raises `NotImplementedError`. Write your version
there and run the check cell below it: the check tests *your* function and prints ✅ only when yours passes. Until you
write it, the notebook stops at that check with `NotImplementedError`. To read on first (the worked examples and the
primer's numbers in section 4 run on the reference solutions), set `USE_REFERENCE = True` in the setup cell: each
unattempted check then runs the reference solution and prints ✗ "not attempted", never ✅. The reference solutions
are in the collapsed cell after the setup; open it only after trying.

In [ ]:
import hashlib, pickle
from dataclasses import dataclass

BLOCK_SIZE = 4                                   # vLLM's default is 16

def cdiv(a, b):
    return -(-a // b)

def H(obj) -> bytes:                             # vllm/utils/hashing.py: sha256
    return hashlib.sha256(pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)).digest()

NONE_HASH = H("vllm-none-hash")                  # init_none_hash, fixed default seed

def hash_block_tokens(parent, tokens, extra_keys=None) -> bytes:
    return H((parent or NONE_HASH, tuple(tokens), extra_keys))

def block_hashes(tokens, lora_name=None, cache_salt=None) -> list:
    """Full blocks only. LoRA name on every block; cache salt on the first block only."""
    out, parent = [], None
    for start in range(0, len(tokens) - BLOCK_SIZE + 1, BLOCK_SIZE):
        keys = ([lora_name] if lora_name else []) + ([cache_salt] if start == 0 and cache_salt else [])
        parent = hash_block_tokens(parent, tokens[start:start + BLOCK_SIZE], tuple(keys) or None)
        out.append(parent)
    return out

@dataclass(eq=False)
class Block:                                     # KVCacheBlock
    block_id: int
    ref_cnt: int = 0
    block_hash: bytes = None
    prev: "Block" = None
    next: "Block" = None

class FreeQueue:                                 # FreeKVCacheBlockQueue: fake head and tail, O(1) remove
    def __init__(self, blocks):
        self.head, self.tail, self.num_free = Block(-1), Block(-1), 0
        self.head.next, self.tail.prev = self.tail, self.head
        self.append_n(blocks)
    def _insert_after(self, where, b):
        b.prev, b.next = where, where.next
        where.next.prev = where.next = b
        self.num_free += 1
    def append_n(self, blocks):                  # to the tail, in order: reused last
        for b in blocks: self._insert_after(self.tail.prev, b)
    def prepend_n(self, blocks):                 # to the head, in order: reused first
        for b in reversed(blocks): self._insert_after(self.head, b)
    def remove(self, b):                         # a prefix hit takes a block out of the middle
        b.prev.next, b.next.prev = b.next, b.prev
        b.prev = b.next = None
        self.num_free -= 1
    def popleft(self):
        assert self.head.next is not self.tail, "no free blocks"
        b = self.head.next; self.remove(b); return b
    def ids(self):
        out, b = [], self.head.next
        while b is not self.tail: out.append(b.block_id); b = b.next
        return out

IMPL, REFERENCE = {}, {}                         # exercise name -> implementation in use / reference solution
USE_REFERENCE = False                            # True: run past exercises you have not written, on the reference

class BlockPool:
    def __init__(self, num_blocks):
        self.blocks = [Block(i) for i in range(num_blocks)]
        self.free = FreeQueue(self.blocks)
        self.cached = {}                         # BlockHashToBlockMap: hash -> {block_id: block}, no dedup
        self.null_block = self.free.popleft()    # block 0 is the reserved null block
    def lookup(self, h):
        d = self.cached.get(h)
        return next(iter(d.values())) if d else None
    def cache(self, b, h):
        b.block_hash = h
        self.cached.setdefault(h, {})[b.block_id] = b
    def get_new_blocks(self, n):                 # pop from the head; evict a hash the block still carries
        out = []
        for _ in range(n):
            b = self.free.popleft()
            if b.block_hash is not None:         # _maybe_evict_cached_block
                d = self.cached[b.block_hash]; del d[b.block_id]
                if not d: del self.cached[b.block_hash]
                b.block_hash = None
            b.ref_cnt = 1; out.append(b)
        return out
    def touch(self, blocks):                     # hit blocks: leave the free queue if unreferenced, ref_cnt += 1
        for b in blocks:
            if b.ref_cnt == 0: self.free.remove(b)
            b.ref_cnt += 1
    def free_blocks(self, ordered):              # callers pass reversed(request_blocks); body = exercise 3
        IMPL["free_blocks"](self, list(ordered))
    def get_usage(self):                         # what vllm:kv_cache_usage_perc reports
        return 1 - self.free.num_free / (len(self.blocks) - 1)

def longest_hit(pool, tokens, **extra):          # exercise 1
    return IMPL["longest_hit"](pool, tokens, **extra)

def can_admit(pool, num_tokens, hit_blocks, watermark_blocks=0):   # exercise 4
    return IMPL["can_admit"](pool, num_tokens, hit_blocks, watermark_blocks)

def admit(pool, tokens, **extra):
    """get_computed_blocks + allocate_slots(full_sequence_must_fit=True) for a waiting request.
    Returns (blocks, hit_tokens), or None when the whole sequence does not fit."""
    hit = longest_hit(pool, tokens, **extra)
    if not can_admit(pool, len(tokens), hit):
        return None
    pool.touch(hit)
    blocks = hit + pool.get_new_blocks(cdiv(len(tokens), BLOCK_SIZE) - len(hit))
    for b, h in zip(blocks, block_hashes(tokens, **extra)):
        if b.block_hash is None: pool.cache(b, h)   # cache_blocks at schedule time: full blocks only
    return blocks, len(hit) * BLOCK_SIZE

def not_attempted(what):
    """Stop here, or with USE_REFERENCE = True say so and carry on (never a ✅: it is not your answer)."""
    if not USE_REFERENCE:
        raise NotImplementedError(f"{what}: not attempted yet. Write yours above and re-run this cell, "
                                  "or set USE_REFERENCE = True in the setup cell to read on with the reference.")
    print(f"✗ {what}: not attempted yet (USE_REFERENCE = True: the notebook uses the reference until you write yours)")

def check(name, test, yours):
    """Run `test` against your implementation. Not written yet: stop, or run the reference and say so."""
    try:
        IMPL[name] = yours
        test(yours)
        print(f"✅ {name}: your implementation passes")
    except NotImplementedError:
        IMPL[name] = REFERENCE[name]
        not_attempted(name)
        test(REFERENCE[name])
        print(f"   {name}: the reference solution passes this check")

print("ready: block size", BLOCK_SIZE)

In [ ]:
#@title Reference solutions (collapsed: open only after trying the exercises) { display-mode: "form" }
def _ref_longest_hit(pool, tokens, **extra):
    hit = []
    for h in block_hashes(tokens, **extra)[: (len(tokens) - 1) // BLOCK_SIZE]:   # cap: num_tokens - 1
        b = pool.lookup(h)
        if b is None:
            break                                # a miss implies a miss for every later block
        hit.append(b)
    return hit

def _ref_free_blocks(pool, ordered):
    first, last = [], []
    for b in ordered:
        b.ref_cnt -= 1
        if b.ref_cnt == 0:
            (last if b.block_hash is not None else first).append(b)
    pool.free.prepend_n(first)                   # uncached: "LIFO reuse ... for better GPU locality"
    pool.free.append_n(last)                     # cached: "FIFO reuse ... for LRU eviction behavior"

def _ref_can_admit(pool, num_tokens, hit_blocks, watermark_blocks=0):
    new = cdiv(num_tokens, BLOCK_SIZE) - len(hit_blocks)
    evictable = sum(b.ref_cnt == 0 for b in hit_blocks)     # touching them takes them out of the free queue
    return new + evictable + watermark_blocks <= pool.free.num_free

REFERENCE.update(longest_hit=_ref_longest_hit, free_blocks=_ref_free_blocks, can_admit=_ref_can_admit)
IMPL.update(REFERENCE)                           # until you write yours
print("reference solutions loaded")

## 1. A hash names the whole prefix (worked example)

Each hash is computed from its parent hash, so editing one token changes that block's hash and every hash after it,
and leaves the blocks before it untouched. Only full blocks are hashed.

In [ ]:
prompt = list(range(100, 116))            # 16 tokens = 4 full blocks
edited = prompt.copy(); edited[5] = 999  # a token inside block 1
same = [a == b for a, b in zip(block_hashes(prompt), block_hashes(edited))]
print("block hash unchanged?", same)
assert same == [True, False, False, False]
assert len(block_hashes(prompt[:15])) == 3   # the partial fourth block has no hash yet
print("✅ the edit in block 1 changes blocks 1-3; a partial block is not hashed")

## 2. Extra keys isolate adapters and tenants (worked example)

The LoRA adapter name is an extra key on every block; `cache_salt` is added to the first block only, yet it still
changes every hash, because every later hash chains from the first. Two tenants with the same prompt but different
salts, or the same prompt under two adapters, never share a block.

In [ ]:
plain = block_hashes(prompt)
lora = block_hashes(prompt, lora_name="adapter-a")
salted = block_hashes(prompt, cache_salt="tenant-42")
assert all(a != b for a, b in zip(plain, lora))
assert all(a != b for a, b in zip(plain, salted))
print("✅ a LoRA name or a first-block salt changes every block hash")

## Exercise 1: the longest cache hit

`get_computed_blocks` walks the prompt's block hashes in order and stops at the first miss (a miss on block *i* means
every later block misses too, because their hashes chain through *i*). It never returns more than
`(num_tokens − 1) // BLOCK_SIZE` blocks: the last prompt token must run through the model to produce the logits the
first output token is sampled from, and hits are whole blocks.

Write `my_longest_hit(pool, tokens, **extra)`: return the list of cached `Block`s (use `pool.lookup(h)`, which
returns `None` on a miss) for the longest cached prefix of `tokens`, with that cap. Before running the check,
predict: a 17-token prompt was admitted once; how many tokens hit when the identical 17 tokens arrive again?

In [ ]:
def my_longest_hit(pool, tokens, **extra):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_longest_hit(impl):
    pool = BlockPool(num_blocks=16)
    assert admit(pool, prompt)[1] == 0                           # empty cache
    assert len(impl(pool, prompt)) == 3                          # identical 16 tokens: (16 - 1) // 4 = 3 blocks
    assert len(impl(pool, prompt + [7, 7, 7, 7, 7])) == 4        # 21 tokens sharing all 4 full blocks
    changed = prompt.copy(); changed[9] = 999                    # block 2 differs
    assert len(impl(pool, changed)) == 2
    assert len(impl(pool, prompt, lora_name="adapter-a")) == 0   # another adapter: nothing shared
    p17 = list(range(300, 317)); admit(pool, p17)
    assert len(impl(pool, p17)) == 4                             # (17 - 1) // 4 = 4: 16 tokens hit, 1 computed

check("longest_hit", test_longest_hit, my_longest_hit)

## Exercise 2: predict the free queue

`KVCacheManager.free` hands a request's blocks to `BlockPool.free_blocks` in **reverse** order; blocks without a hash
(the partial last block, which no request can ever hit) go to the **head** of the free queue, blocks with a hash go to
the **tail**. Allocation pops from the head. (The mini engine sends every block to the back; this is one of the
places vLLM differs.)

Request X has a 14-token prompt (three full blocks and one partial) and gets blocks 1–4 from a fresh pool of 14 blocks
(1–13 usable). X finishes. Then an unrelated request Y needs 11 blocks. Fill in both predictions before running the
check.

In [ ]:
# Replace None with your predictions.
predicted_queue_after_x = None     # pool.free.ids(), head to tail, right after X finishes, e.g. [5, 6, ...]
predicted_x_survivors = None       # after Y: is each of X's 3 full blocks still cached? e.g. [True, False, False]

In [ ]:
missing = [n for n, v in (("queue", predicted_queue_after_x), ("survivors", predicted_x_survivors)) if v is None]
if missing:                                          # stop before the answer is printed
    not_attempted("exercise 2 predictions (" + " and ".join(missing) + ")")

pool = BlockPool(num_blocks=14)
x_prompt = list(range(200, 214))                     # 14 tokens
x_blocks, _ = admit(pool, x_prompt)
assert [b.block_id for b in x_blocks] == [1, 2, 3, 4]
pool.free_blocks(reversed(x_blocks))
queue_after_x = pool.free.ids()
y_blocks, _ = admit(pool, list(range(500, 544)))     # 44 unrelated tokens = 11 blocks
survivors = [pool.lookup(h) is not None for h in block_hashes(x_prompt)]
print("free queue after X finishes:", queue_after_x)
print("Y got", [b.block_id for b in y_blocks], "| X's full blocks still cached (root first):", survivors)

assert queue_after_x == [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 3, 2, 1]
assert survivors == [True, True, False]
assert len(longest_hit(pool, x_prompt)) * BLOCK_SIZE == 8
for name, predicted, actual in (("queue", predicted_queue_after_x, queue_after_x),
                                ("survivors", predicted_x_survivors, survivors)):
    if predicted is not None:
        assert predicted == actual, f"your {name} prediction {predicted} != {actual}"
        print(f"  your {name} prediction matches")
print(("✅ " if not missing else "   ") + "X's partial block went first, then its chain tail; its root survives for the next hit")

## Exercise 3: implement `free_blocks`, and see LRU refresh

Write `my_free_blocks(pool, ordered)`. `ordered` is a request's blocks, already reversed (tail first). Decrement each
block's `ref_cnt`; the blocks that reach zero go back to `pool.free`: those **without** a hash to the head with
`pool.free.prepend_n(list)`, those **with** a hash to the tail with `pool.free.append_n(list)`, each list in the order
you met the blocks. A block still referenced by another request stays out of the queue.

The check also tests the property that makes the queue an LRU: X and Z finish (X first), then X's prompt is hit
again (the hit pulls X's blocks out of the middle of the queue) and X finishes again. X's blocks are now the *most*
recently freed, so Z's chain is evicted first.

In [ ]:
def my_free_blocks(pool, ordered):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_free_blocks(impl):
    # The worked case of exercise 2.
    pool = BlockPool(num_blocks=14)
    xb, _ = admit(pool, list(range(200, 214)))
    impl(pool, list(reversed(xb)))
    assert pool.free.ids() == [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 3, 2, 1]

    # A shared block is not freed while another request holds it.
    pool = BlockPool(num_blocks=10)
    a = list(range(100, 109))                         # 9 tokens: blocks 1, 2 full, block 3 partial
    ab, _ = admit(pool, a)
    bb, hit = admit(pool, a[:8] + [555])              # shares both full blocks (cap (9-1)//4 = 2)
    assert hit == 8 and [b.ref_cnt for b in ab[:2]] == [2, 2]
    impl(pool, list(reversed(ab)))
    assert [b.ref_cnt for b in ab[:2]] == [1, 1] and pool.free.ids()[0] == 3   # only A's partial block is free

    # LRU refresh: X, then Z finish; X is hit again and finishes again; Z is now older and goes first.
    pool = BlockPool(num_blocks=12)                   # blocks 1..11
    x, z = list(range(200, 208)), list(range(600, 608))   # two full blocks each
    xb, _ = admit(pool, x); zb, _ = admit(pool, z)    # X: 1, 2   Z: 3, 4
    impl(pool, list(reversed(xb))); impl(pool, list(reversed(zb)))
    assert pool.free.ids() == [5, 6, 7, 8, 9, 10, 11, 2, 1, 4, 3]
    xb2, hit = admit(pool, x + [9])                   # 9 tokens: hits X's 2 blocks, 1 new block (5) for the partial
    assert hit == 8 and [b.block_id for b in xb2] == [1, 2, 5]
    impl(pool, list(reversed(xb2)))
    assert pool.free.ids() == [5, 6, 7, 8, 9, 10, 11, 4, 3, 2, 1]
    pool.get_new_blocks(9)                            # pops 5..11, then Z's 4 and 3
    assert all(pool.lookup(h) is None for h in block_hashes(z))
    assert all(pool.lookup(h) is not None for h in block_hashes(x))

check("free_blocks", test_free_blocks, my_free_blocks)

## 3. What `vllm:kv_cache_usage_perc` does not show (worked example)

`BlockPool.get_usage` is `1 − free / (num_blocks − 1)`, and cached blocks in the free queue count as free. A replica
can report 0% usage while holding a warm prefix cache, which is why a cache-aware router needs block hashes (KV
events), not this gauge.

In [ ]:
pool = BlockPool(num_blocks=9)             # 8 usable blocks
blocks, _ = admit(pool, prompt)            # 16 tokens = 4 blocks
usage_running = pool.get_usage()
pool.free_blocks(reversed(blocks))
usage_after, still_cached = pool.get_usage(), len(pool.cached)
hit_after = len(longest_hit(pool, prompt)) * BLOCK_SIZE
print(f"usage while running {usage_running:.2f}, after finishing {usage_after:.2f}, "
      f"cached blocks {still_cached}, next hit {hit_after} tokens")
assert (usage_running, usage_after, still_cached, hit_after) == (0.5, 0.0, 4, 12)
print("✅ usage fell to 0 while all four blocks stayed cached and reusable")

## Exercise 4: the admission gate, and why a preempted request waits

`allocate_slots(..., full_sequence_must_fit=True)` admits a waiting request only if its **whole current sequence**
(prompt, plus the outputs it already produced if it was preempted) fits. `get_num_blocks_to_allocate` counts the new
blocks, and it also counts every hit block that currently sits in the free queue, because touching it removes it
from the queue ("If a computed block is an eviction candidate ... we must count it in the free-capacity check").

Write `my_can_admit(pool, num_tokens, hit_blocks, watermark_blocks=0)`: return `True` iff

    (cdiv(num_tokens, BLOCK_SIZE) − len(hit_blocks)) + #(hit blocks with ref_cnt == 0) + watermark_blocks ≤ pool.free.num_free

The check then replays primer Section 3.7 at small scale with your gate: two requests with 8-token prompts decode
in lockstep in a pool of 9 usable blocks; the second is preempted; a third, short request arrives afterwards.
*Predict first:* when is the preempted request re-admitted, and when is the short one?

In [ ]:
def my_can_admit(pool, num_tokens, hit_blocks, watermark_blocks=0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def replay_preemption(num_blocks=10, prompt_len=8, r1_new_tokens=20, r3_arrives=10):
    """Synchronous replay of Scheduler.schedule + update_from_output for three requests (FCFS).
    Each step: (1) running pass: each running request computes one token; the token at position p needs
    block p // BLOCK_SIZE; if none is free, preempt the newest running request (running[-1]), and stop the
    pass if the request preempted itself; (2) waiting pass, skipped in a step that preempted: admit from the
    head with `admit` (your gate) and stop at the first request that does not fit; (3) after the step: advance,
    cache blocks that became full, sample a token, free finished requests."""
    pool = BlockPool(num_blocks)
    reqs, running, waiting, preemptions, trace, admitted_at = {}, [], [], 0, [], {}
    for name, base in (("R1", 1000), ("R2", 2000)):
        toks = list(range(base, base + prompt_len))
        blocks, _ = admit(pool, toks)
        reqs[name] = dict(tokens=toks + [base + 500], blocks=blocks, computed=prompt_len)
        running.append(name)
    step = 0
    while running or waiting:
        step += 1
        if step == r3_arrives:                             # a short request joins the queue behind R2
            reqs["R3"] = dict(tokens=[3000, 3001, 3002], blocks=[], computed=0)
            waiting.append("R3")
        scheduled, preempted_now = [], False
        for name in list(running):                          # (1) running pass
            if name not in running:
                continue
            r = reqs[name]
            if r["computed"] // BLOCK_SIZE >= len(r["blocks"]):
                while pool.free.num_free == 0:
                    victim = running.pop()
                    pool.free_blocks(reversed(reqs[victim]["blocks"]))
                    reqs[victim].update(blocks=[], computed=0)
                    waiting.insert(0, victim); preemptions += 1; preempted_now = True
                    if victim == name:
                        break
                if name not in running:
                    break
                r["blocks"] += pool.get_new_blocks(1)
            scheduled.append(name)
        if not preempted_now:                               # (2) waiting pass
            while waiting:
                name = waiting[0]; r = reqs[name]
                hit = longest_hit(pool, r["tokens"])
                result = admit(pool, r["tokens"])
                trace.append((step, name, len(hit), pool.free.num_free if result is None else None, result is not None))
                if result is None:
                    break                                   # the head does not fit: nobody behind it is admitted
                r["blocks"], hit_tokens = result
                r["recomputed"] = len(r["tokens"]) - hit_tokens
                r["computed"] = len(r["tokens"]) - 1        # the prefill runs this step, like a decode of the rest
                waiting.pop(0); running.append(name); scheduled.append(name); admitted_at[name] = step
        for name in scheduled:                              # (3) execute + update_from_output
            r = reqs[name]
            r["computed"] += 1
            full = r["computed"] // BLOCK_SIZE
            hashes = block_hashes(r["tokens"][: r["computed"]])
            for i in range(full):
                if r["blocks"][i].block_hash is None:
                    pool.cache(r["blocks"][i], hashes[i])
            r["tokens"].append(len(r["tokens"]) + 7 * (name == "R2"))   # the sampled token
            if name == "R1" and r["computed"] >= prompt_len + r1_new_tokens:
                pool.free_blocks(reversed(r["blocks"])); running.remove("R1"); admitted_at["R1 finished"] = step
        if "R2" in admitted_at and "R3" in admitted_at:
            break
    return dict(preemptions=preemptions, trace=trace, admitted_at=admitted_at, recomputed=reqs["R2"]["recomputed"])

def test_can_admit(impl):
    pool = BlockPool(num_blocks=10)                        # 9 usable
    a = list(range(100, 124))                              # 24 tokens = 6 blocks
    ab, _ = admit(pool, a)
    assert impl(pool, 12, []) is True                      # 3 new <= 3 free
    assert impl(pool, 13, []) is False                     # 4 > 3
    assert impl(pool, 12, [], 1) is False                  # watermark
    assert impl(pool, 25, ab[:6]) is True                  # held hits cost nothing: 1 new <= 3
    pool.free_blocks(reversed(ab))                         # a finishes: its 6 blocks are cached and free
    assert impl(pool, 25, ab[:6]) is True                  # 1 new + 6 evictable = 7 <= 9
    assert impl(pool, 37, ab[:6], 3) is False              # 4 + 6 + 3 = 13 > 9

    out = replay_preemption()
    print("preemptions:", out["preemptions"], "| admitted at step:", out["admitted_at"])
    print("waiting-pass trace (step, head, hit blocks, free if refused, admitted):")
    for row in out["trace"]:
        print("  ", row)
    assert out["preemptions"] == 1                         # one preemption, no ping-pong
    r2_rows = [row for row in out["trace"] if row[1] == "R2"]
    assert [row[2] for row in r2_rows] == [4, 4, 4, 3, 3, 3, 3, 2, 2, 2, 2, 2]   # tail evicted every 4 tokens
    assert all(not ok for (_, _, _, _, ok) in r2_rows[:-1]) and r2_rows[-1][4]
    assert out["admitted_at"]["R2"] == out["admitted_at"]["R1 finished"] + 1   # only after R1 frees its blocks
    assert out["admitted_at"]["R3"] == out["admitted_at"]["R2"]               # R3 fits from step 10, but waits
    assert out["recomputed"] == 9                          # 17 tokens, 2 blocks (8 tokens) still cached

check("can_admit", test_can_admit, my_can_admit)

What the replay shows, in the primer's terms: at the preemption R2 held four full blocks (16 tokens computed, 17
known). Re-admission needs `cdiv(17, 4) = 5` blocks; four of them are its own cached blocks, but those are *in* the
free queue, so they count, and 1 + 4 = 5 is more than the 4 free blocks that exist while R1 runs. Every 4 tokens R1
takes the queue head, which is R2's chain tail. R2 comes back only when R1 finishes, re-hits the 2 surviving blocks and
recomputes 9 of its 17 tokens; and R3, which would have fit all along, waited behind it because the waiting pass stops
at the first request that does not fit. At full scale (16-token blocks, 2,000-token prompts, 299 usable blocks) the
numbers are 150 needed against 149 free (Section 3.7 of the primer, recomputed in the next section).

Why the evictable-hit term matters: without it, the gate would admit R2 at step 10 (1 new block against 4 free), and
the allocation would then run the free queue dry, because those 4 "free" blocks are exactly the hit blocks R2 is
about to take out of it. In vLLM the same count appears twice in `allocate_slots` (the full-sequence check and the
per-step check), so the request is refused instead.

## 4. The primer's worked numbers, recomputed

Every number in the primer's worked examples, computed here and asserted, so the primer, the serving lab and this
notebook cannot drift apart silently. The KV budget (primer 4.7) and quantization (8.2) rows use the serving lab's
`servelab.sizing` from [`../../serving-engine/vllm-serving-lab/`](../../serving-engine/vllm-serving-lab/); the
roofline peaks are the layer-01 `roofline.specs` values (verify against datasheets). These are calculations from
stated assumptions, not measurements.

In [ ]:
import math, pathlib, sys

# §3.5: four requests through six steps (block_size 16)
B = 16
assert [cdiv(n, B) for n in (3000, 500, 6000)] == [188, 32, 375]
assert cdiv(2046, B) == 128 and cdiv(4092, B) - 128 == 128 and 375 - cdiv(4092, B) == 119
assert cdiv(6000 + 1, B) - 375 == 1                              # step 6: C's first decode opens block 375
assert cdiv(3003 + 1, B) == 188 and cdiv(503 + 1, B) == 32         # A and B need nothing in step 6

# §3.7: the full-scale preemption case
usable = 300 - 1
free = usable - 2 * cdiv(2000, B)                                  # 49 after both prefills
assert free == 49 and free - 2 * 24 == 1                           # 24 decode blocks each leave 1
r2_blocks, r2_num_tokens = 125 + 24, 2000 + 24 * B + 1             # 149 full blocks, 2,385 tokens known
free_after = usable - (125 + 24 + 1)                               # R1 took the last block: 149 free (all R2's)
hit = (r2_num_tokens - 1) // B
need = (cdiv(r2_num_tokens, B) - hit) + hit                        # new + evictable hits
assert (hit, need, free_after) == (149, 150, 149) and need > free_after

# §4.4: prefix-hit table
hit_blocks = 1000 // B
assert hit_blocks == 62 and 1200 - hit_blocks * B == 208 and round(100 * hit_blocks * B / 1200, 1) == 82.7
assert (1200 - 1) // B * B == 1184 and round(100 * 1184 / 1200, 1) == 98.7

# §5.5: default CUDA-graph capture sizes
def capture_sizes(max_num_seqs, max_num_batched_tokens, k=0, ceiling=512):
    q = 1 + k
    cap = min(max_num_seqs * q * 2, ceiling)
    decode = []
    if q > 1:
        m = min(max_num_seqs, ceiling)
        counts = sorted(set([n for n in (1, 2, 4) if n <= m] + list(range(8, min(m + 1, 256), 8))
                            + list(range(256, m + 1, 16)) + [m]))
        decode = [n * q for n in counts if n * q <= cap]
    elif max_num_seqs <= cap:
        decode = [max_num_seqs]
    cap = min(max_num_batched_tokens, cap)
    sizes = [n for n in (1, 2, 4) if n <= cap] + list(range(8, min(cap + 1, 256), 8)) + list(range(256, cap + 1, 16))
    if max_num_batched_tokens <= cap:
        sizes.append(max_num_batched_tokens)
    return sorted(set(sizes + [d for d in decode if d <= max_num_batched_tokens]))
assert len(capture_sizes(256, 2048)) == 51 and capture_sizes(256, 2048)[-1] == 512
extra = sorted(set(capture_sizes(256, 2048, k=2)) - set(capture_sizes(256, 2048)))
assert extra == [3, 6, 12, 264, 312, 360, 408, 456, 504] and len(capture_sizes(256, 2048, k=2)) == 60
assert capture_sizes(256, 2048, k=3) == capture_sizes(256, 2048)       # every n x 4 is already on the grid

# §7.3: grammar bitmask size for a 128,256-token vocabulary
words = cdiv(128256, 32)
assert (words, words * 4, round(256 * words * 4 / 1e6, 1), round(1024 * words * 4 / 1e6, 1)) == (4008, 16032, 4.1, 16.4)

# §7.4: expected tokens per target step
alpha, k = 0.7, 3
assert round((1 - alpha ** (k + 1)) / (1 - alpha), 2) == 2.53

# §8.1: roofline time of Llama-3.1-8B's down_proj (K=14336, N=4096) on an L4
L4 = dict(bw=0.30e12, bf16=121e12, fp8=242.5e12)                  # roofline.specs "l4" (verify)
def gemm_us(M, w_bytes, a_bytes, peak, dev=L4, K=14336, N=4096):
    flops, byts = 2 * M * K * N, K * N * w_bytes + M * K * a_bytes + M * N * 2
    return max(flops / dev[peak], byts / dev["bw"]) * 1e6
W4 = 0.5 + 2.5 / 128                                                # 4-bit weights + fp16 scale and 4-bit zero per 128
table = {M: [round(gemm_us(M, 2, 2, "bf16")), round(gemm_us(M, W4, 2, "bf16")), round(gemm_us(M, 1, 1, "fp8"))]
         for M in (1, 256, 2048)}
print("§8.1 GEMM µs (BF16, W4A16, W8A8):", table)
assert table == {1: [392, 102, 196], 256: [423, 248, 215], 2048: [1988, 1988, 992]}

# §10.3: moving 4,000 tokens of Llama-3.1-8B KV
kv = 4000 * 131072
assert kv == 524_288_000
assert [round(kv / g / 1e9 * 1e3, 1) for g in (450, 50, 12.5, 1.25)] == [1.2, 10.5, 41.9, 419.4]
print("✅ §3.5, §3.7, §4.4, §5.5, §7.3, §7.4, §8.1 and §10.3 numbers match the primer")

In [ ]:
def find_serving_lab():
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "serving-engine" / "vllm-serving-lab",
                     base / "04-inference-engine" / "serving-engine" / "vllm-serving-lab"):
            if (cand / "servelab" / "sizing.py").exists():
                return cand
    return None

lab = find_serving_lab()
if lab is None:
    print("serving lab not found next to this notebook: skipping the §4.7 and §8.2 checks "
          "(they need 04-inference-engine/serving-engine/vllm-serving-lab from this repository)")
else:
    sys.path.insert(0, str(lab))
    from servelab import sizing
    GiB, MODEL = 1024 ** 3, "llama-3.1-8b-instruct"
    H100 = dict(max_num_batched_tokens=8192, max_num_seqs=1024)
    m = sizing.load_config(MODEL)
    assert sizing.param_count(m).total == 8_030_261_248
    assert sizing.kv_bytes_per_token(m) == 131_072 and 16 * sizing.kv_bytes_per_token(m) == 2 * 1024 ** 2

    # §4.7 budget table
    l4, h100 = sizing.size(MODEL, "L4", max_model_len=8192), sizing.size(MODEL, "H100-80GB", max_model_len=8192, **H100)
    for r in (l4, h100):
        print(f"{r.gpu}: requested {r.requested_bytes / GiB:.2f} GiB, weights {r.weights_bytes / GiB:.2f}, "
              f"overheads { {k: round(v / GiB, 2) for k, v in r.overhead_bytes.items()} }, "
              f"KV {r.kv_budget_bytes / GiB:.2f} GiB, {r.num_blocks:,} blocks, {r.kv_capacity_tokens:,} tokens")
    assert (round(l4.requested_bytes / GiB, 2), round(l4.weights_bytes / GiB, 2)) == (20.69, 14.96)
    assert (round(l4.kv_budget_bytes / GiB, 2), l4.num_blocks, l4.kv_capacity_tokens) == (4.62, 2363, 37808)
    assert (round(h100.kv_budget_bytes / GiB, 2), h100.num_blocks, h100.kv_capacity_tokens) == (55.95, 28648, 458368)
    conc = lambda gpu, L, **kw: round(sizing.size(MODEL, gpu, max_model_len=L, **kw).max_concurrency, 2)
    assert [conc("L4", L) for L in (8192, 16384, 32768)] == [4.62, 2.31, 1.15]
    assert [conc("H100-80GB", L, **H100) for L in (8192, 32768, 131072)] == [55.95, 13.99, 3.5]
    default_l4 = sizing.size(MODEL, "L4")                            # max_model_len = 131,072
    assert not default_l4.fits and default_l4.blocks_per_request * default_l4.bytes_per_block == 16 * GiB
    assert default_l4.estimated_max_model_len == 37808               # what `--max-model-len auto` approaches
    fp8kv = [sizing.size(MODEL, g, max_model_len=8192, kv_cache_dtype="fp8", **kw)
             for g, kw in (("L4", {}), ("H100-80GB", H100))]
    assert [(r.num_blocks, r.kv_capacity_tokens) for r in fp8kv] == [(4727, 75632), (57297, 916752)]

    # §8.2 quantization table: weight bytes, streamed bytes, floors, freed blocks on the L4
    embed_bytes = 128256 * 4096 * 2                                  # the input embedding is gathered, not streamed
    rows = {}
    for label, q in (("BF16", None), ("FP8", "fp8"), ("INT4", "gptq")):
        w = int(sizing.weight_bytes(m, quantization=q))
        blocks = sizing.size(MODEL, "L4", max_model_len=8192, quantization=q).num_blocks
        rows[label] = (round(w / 1e9, 2), round((w - embed_bytes) / 1e9, 2), round((w - embed_bytes) / 300e9 * 1e3, 1),
                       round((w - embed_bytes) / 3.35e12 * 1e3, 2), blocks - l4.num_blocks)
    print("§8.2 (GB, streamed GB, L4 ms, H100 ms, extra blocks):", rows)
    assert rows == {"BF16": (16.06, 15.01, 50.0, 4.48, 0), "FP8": (9.08, 8.03, 26.8, 2.4, 3328),
                    "INT4": (5.73, 4.68, 15.6, 1.4, 4927)}
    print("✅ §4.7 and §8.2 numbers match the primer and the serving lab's sizing model")

## In a design review

**Explain it in two minutes.** "vLLM hashes each full 16-token block together with its parent's hash, so a hash
names the whole prefix; LoRA names and a tenant salt are folded in, which isolates adapters and tenants. When a
request finishes, its blocks go back to a doubly linked free list in reverse order: the partial block to the head,
because nothing can ever hit it, and the hashed blocks to the tail, so they are the last to be reused. Allocation pops
the head, so cached blocks are evicted in LRU order and a chain loses its tail before its root, which is the part the
next request most likely shares; a later hit pulls a block out of the middle of the list in constant time and
refreshes it. Admission checks that the whole current sequence fits, and cached hits sitting in the free list count
as capacity to be taken. That is why a request preempted under pressure usually cannot return until the request that
displaced it finishes, and why everything queued behind it waits too."

**Drills.**

1. *Why evict a chain's tail before its root?* The root (system prompt, tool schemas) is shared by the most future
   requests; the tail is specific to one conversation. Freeing in reverse and appending puts the root last in line.
2. *A router reads `kv_cache_usage_perc = 0` on a replica. Is its cache cold?* Not necessarily: usage counts only
   blocks referenced by live requests (section 3). Cache affinity needs block hashes (KV events), not this gauge.
3. *Why does the partial last block go to the head?* It has no hash, so no request can hit it; reusing it first keeps
   useful cached blocks alive longer (and, per the source comment, helps GPU locality).
4. *A preempted request's blocks are all still cached. Why is it not re-admitted next step?* Its re-admission must
   fit its whole sequence, and its own cached blocks count as capacity because they sit in the free queue; with the
   survivor holding the rest of the pool, it is one block short, and the survivor keeps eating its chain from the
   tail (exercise 4).